In [432]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, auc

In [433]:
seed = 42

root_path = "/home/stefan/kits-locala/churn"

# Data

In [434]:
train_df = pd.read_csv(f"{root_path}/train.csv")
test_df = pd.read_csv(f"{root_path}/test.csv")

test_df.head()

,SampleID,Age,Avg Monthly GB Download,Avg Monthly Long Distance Charges,City,CLTV,Contract,Country,Customer ID,Dependents,...,Total Refunds,Total Revenue,Under 30,Unlimited Data,Zip Code,Ping Score,Avg Speed,TV Type,Link Quality Index,Total Short Distance Charges
0,5635,50,5,41.38,Wilseyville,3167,Month-to-Month,United States,0988-JRWWP,0,...,0.0,264.54,0,1,95257,60.014827,99.619687,Satellite Receiver,49.361872,1.760734
1,5636,54,10,24.17,Newcastle,3351,Month-to-Month,United States,7718-RXDGG,0,...,0.0,1561.15,0,0,95658,73.310555,103.638643,Satellite Receiver,56.438374,20.033471
2,5637,35,0,34.79,Bolinas,5890,Month-to-Month,United States,6121-VZNQB,0,...,0.0,53.89,0,0,94924,51.986775,100.025968,Satellite Receiver,14.242867,2.774798
3,5638,57,18,20.67,Upland,5413,Month-to-Month,United States,9552-TGUZV,0,...,0.0,823.46,0,1,91784,29.075805,98.211552,Satellite Receiver,57.958135,10.305294
4,5639,31,0,22.12,Kyburz,3663,Month-to-Month,United States,1963-VAUKV,0,...,0.0,42.52,0,0,95720,57.144365,100.635870,Satellite Receiver,16.206531,-0.315167


# Subtask 1

In [435]:
# FinancialRiskScore = (Monthly Charge > 70) + (Total Extra Data Charges > 10)

test_df["FinancialRiskScore"] = (test_df["Monthly Charge"] > 70).astype(np.int32) + (
    test_df["Total Extra Data Charges"] > 10
).astype(np.int32)

train_df["FinancialRiskScore"] = (train_df["Monthly Charge"] > 70).astype(np.int32) + (
    train_df["Total Extra Data Charges"] > 10
).astype(np.int32)

subtask1 = test_df["FinancialRiskScore"]

# Subtask 2

In [436]:
# ServiceQualityScore = (Avg Speed < 50) + (Ping Score > 80) + (Link Quality Index < 30)

test_df["ServiceQualityScore"] = (
    (test_df["Avg Speed"] < 50).astype(np.int32)
    + (test_df["Ping Score"] > 80).astype(np.int32)
    + (test_df["Link Quality Index"] < 30).astype(np.int32)
)

train_df["ServiceQualityScore"] = (
    (train_df["Avg Speed"] < 50).astype(np.int32)
    + (train_df["Ping Score"] > 80).astype(np.int32)
    + (train_df["Link Quality Index"] < 30).astype(np.int32)
)

subtask2 = test_df["ServiceQualityScore"]

In [437]:
subtask2.describe()

count    1409.000000
mean        0.350603
std         0.526838
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max         2.000000
Name: ServiceQualityScore, dtype: float64

# Subtask 3

In [438]:
train_df.select_dtypes(include='str').nunique().sort_values(ascending=False)

Customer ID       5634
Lat Long          1669
City              1103
Offer                5
TV Type              3
Contract             3
Internet Type        3
Payment Method       3
Gender               2
Country              1
Quarter              1
State                1
dtype: int64

In [439]:
def prep_df(df: pd.DataFrame):
    if "Churn" in df:
        y = df["Churn"]
    else:
        y = None

    df = df.drop(["SampleID", "Customer ID", "City", "Churn"], axis=1, errors='ignore')

    df["lat"] = df["Lat Long"].str.split(", ").apply(lambda x: np.float32(x[0]))
    df["long"] = df["Lat Long"].str.split(", ").apply(lambda x: np.float32(x[1]))

    # 1. impute missiong
    missing_cols = df.columns[df.isna().sum() > 0]
    for col in missing_cols:
        val = df[col].mode() if df[col].dtype == 'str' else df[col].mean()
        df[col] = df[col].fillna(value=val)

    # 2. dummy encoding
    cat_cols = df.select_dtypes(include='str')
    for col in cat_cols:
        if df[col].nunique() > 5:
            df = df.drop([col], axis=1)
            continue
        dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop([col], axis=1)

    if y is not None:
        return df, y
    return df

In [440]:
X_train, y_train = prep_df(train_df)

test_ids = test_df["SampleID"]
test_df = prep_df(test_df)

In [441]:
X_train.isna().sum().sum()

np.int64(0)

In [442]:
X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=seed)

In [443]:
def evaluate(clf):
    cv = cross_val_score(clf, X_train, y_train, scoring='roc_auc', cv=3, n_jobs=-1)
    return cv.mean() - cv.std()

In [444]:
rf = RandomForestClassifier(n_estimators=400, random_state=seed)
evaluate(rf)

np.float64(0.9800931559900845)

In [445]:
model = rf
model.fit(X_train, y_train)

subtask3 = rf.predict_proba(test_df)[:, 1]

# Submission

In [446]:
def build_subtask(sid, ans):
    return pd.DataFrame({"id": test_ids, "subtaskID": sid, "answer": ans})

subtasks = [
    (1, subtask1.astype(np.int32)),
    (2, subtask2.astype(np.int32)),
    (3, subtask3),
]

submission = pd.concat([build_subtask(sid, ans) for sid, ans in subtasks])

In [447]:
submission.head()

,id,subtaskID,answer
0,5635,1,0.0
1,5636,1,1.0
2,5637,1,0.0
3,5638,1,1.0
4,5639,1,0.0


In [448]:
submission.to_csv(f"{root_path}/submission.csv", index=False)